# Case study — Lipidcane → biodiesel (co-product allocation & TEA)

Lipidcane is engineered sugarcane accumulating oil; the biorefinery makes **biodiesel
and ethanol** together (plus electricity). Multiple saleable products make this the
natural case for **allocation** — the same total impact splits differently by mass,
energy, or economic value. We also read its techno-economics.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from biorefineries import lipidcane as lc

lc.load()
sys = lc.lipidcane_sys
tea = lc.lipidcane_tea
biodiesel, ethanol = lc.biodiesel, lc.ethanol
print("units in system:", len(sys.units))

units in system: 100


## Co-products and techno-economics

In [2]:
bd_kg, et_kg = biodiesel.F_mass, ethanol.F_mass
print(f"biodiesel = {bd_kg:,.0f} kg/hr")
print(f"ethanol   = {et_kg:,.0f} kg/hr")
print(f"IRR             = {tea.solve_IRR():.1%}")
print(f"FCI             = ${tea.FCI/1e6:,.1f} MM")
print(f"MSP biodiesel   = ${tea.solve_price(biodiesel):.4f}/kg")

biodiesel = 8,859 kg/hr
ethanol   = 11,622 kg/hr
IRR             = 20.8%
FCI             = $224.8 MM
MSP biodiesel   = $1.3800/kg


## Allocation between biodiesel and ethanol

Suppose the process carries a total upstream GWP `G` (kg CO2e/hr). How much belongs
to biodiesel? It depends entirely on the allocation basis.

In [3]:
G = 40_000.0     # illustrative total system GWP, kg CO2e/hr

# Mass basis
mass_frac = bd_kg / (bd_kg + et_kg)
# Energy basis (LHV: biodiesel ~37.5 MJ/kg, ethanol ~26.8 MJ/kg)
LHV_bd, LHV_et = 37.5, 26.8
energy_frac = (bd_kg * LHV_bd) / (bd_kg * LHV_bd + et_kg * LHV_et)
# Economic basis (prices: biodiesel 1.38, ethanol 0.72 $/kg)
p_bd, p_et = 1.38, 0.72
econ_frac = (bd_kg * p_bd) / (bd_kg * p_bd + et_kg * p_et)

print("Biodiesel's share of impact and its per-kg footprint:")
for name, frac in [("mass", mass_frac), ("energy", energy_frac), ("economic", econ_frac)]:
    per_kg = G * frac / bd_kg
    print(f"  {name:9s}: {frac:5.1%} of total  ->  {per_kg:.3f} kg CO2e / kg biodiesel")

Biodiesel's share of impact and its per-kg footprint:
  mass     : 43.3% of total  ->  1.953 kg CO2e / kg biodiesel
  energy   : 51.6% of total  ->  2.330 kg CO2e / kg biodiesel
  economic : 59.4% of total  ->  2.681 kg CO2e / kg biodiesel


The spread across bases is the point: allocation choice materially changes the
reported biodiesel footprint, so it must be stated explicitly in any LCA.

In [4]:
results = {
    "biodiesel_kg_hr": round(float(bd_kg), 0),
    "ethanol_kg_hr": round(float(et_kg), 0),
    "IRR": round(float(tea.solve_IRR()), 3),
    "MSP_biodiesel_usd_kg": round(float(tea.solve_price(biodiesel)), 4),
    "biodiesel_mass_alloc_frac": round(float(mass_frac), 3),
    "biodiesel_energy_alloc_frac": round(float(energy_frac), 3),
    "biodiesel_economic_alloc_frac": round(float(econ_frac), 3),
}
print(results)
print("Lipidcane case study complete.")

{'biodiesel_kg_hr': 8859.0, 'ethanol_kg_hr': 11622.0, 'IRR': 0.208, 'MSP_biodiesel_usd_kg': 1.38, 'biodiesel_mass_alloc_frac': 0.433, 'biodiesel_energy_alloc_frac': 0.516, 'biodiesel_economic_alloc_frac': 0.594}
Lipidcane case study complete.
